# Week 6 — Capstone: End-to-End Distributed Training Pipeline

The previous five weeks built the components. This week assembles them into a **production-grade, reusable boilerplate** that trains a 1.3B-parameter GPT model on a multi-node H100 cluster from a single shell command.

## Deliverable

```bash
make launch-cloud PROVIDER=runpod GPU=H100 NUM_GPUS=8 CONFIG=configs/training/gpt_1b3.yaml
```

A green WandB run within 2 minutes; a saved checkpoint within the first epoch; a public dashboard link suitable for the GitHub README.

## Components Combined

| Component                          | From    |
|------------------------------------|---------|
| Roofline-guided dtype selection    | Week 1  |
| DDP with gradient bucketing        | Week 2  |
| Tensor parallelism + 1F1B pipelining | Week 3 |
| ZeRO-3 with CPU offload            | Week 4  |
| Dockerized cloud launch            | Week 5  |
| Spot-resilient checkpointing       | Week 5  |

The result is **`examples/train_gpt_capstone.py`** — a single ~250-line training script that exposes parallelism strategy, precision, and provider as CLI flags.


## 1. Configuration Schema

The training script consumes a YAML config that captures every dial:

```yaml
# configs/training/gpt_1b3.yaml
model:
  name: gpt_1b3
  hidden_size: 2048
  num_layers: 24
  num_heads: 16
  vocab_size: 50257
  seq_length: 2048

training:
  global_batch_size: 1024
  micro_batch_size: 4
  max_steps: 100_000
  lr: 3.0e-4
  weight_decay: 0.1
  grad_clip: 1.0

parallelism:
  data_parallel_size: 8
  tensor_parallel_size: 1
  pipeline_parallel_size: 1

precision:
  dtype: bf16
  loss_scale: dynamic

zero:
  stage: 3
  offload_optimizer: cpu
  offload_param: none

checkpointing:
  interval_steps: 500
  out_dir: /workspace/checkpoints
  keep_last_n: 3
```


In [ ]:
# %% examples/train_gpt_capstone.py — entrypoint sketch
import argparse, yaml, torch, deepspeed
from hpcllmforge.utils.seeding import seed_everything
from hpcllmforge.utils.logging import get_logger
from hpcllmforge.orchestration.checkpoint_manager import CheckpointManager
from hpcllmforge.profiling.throughput_meter import ThroughputMeter

logger = get_logger(__name__)

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--config', required=True)
    p.add_argument('--parallelism', choices=['ddp', 'zero'], default='zero')
    p.add_argument('--deepspeed', default='configs/deepspeed/zero3_offload.json')
    return p.parse_args()

def main():
    args = parse_args()
    cfg = yaml.safe_load(open(args.config))
    seed_everything(cfg.get('seed', 42))

    # 1. Build the model (toy GPT or use HF transformers GPT-2 architecture).
    model = build_gpt(cfg['model'])

    # 2. Build the data loader (your own sharded dataset).
    train_loader = build_dataloader(cfg['training'])

    # 3. Initialise DeepSpeed (handles DDP, ZeRO, FP16, gradient clipping).
    engine, optim, _, _ = deepspeed.initialize(
        model=model, model_parameters=model.parameters(),
        config=args.deepspeed,
    )

    # 4. Set up checkpointing + throughput meter.
    ckpt = CheckpointManager(out_dir=cfg['checkpointing']['out_dir'])
    meter = ThroughputMeter(flops_per_step=estimate_flops(cfg),
                            peak_flops=989e12)  # H100 BF16 peak

    # 5. Train.
    for step, batch in enumerate(train_loader):
        loss = engine(batch)
        engine.backward(loss)
        engine.step()
        meter.tick()
        if step % 20 == 0:
            logger.info('step=%d loss=%.4f mfu=%.2f%%',
                        step, loss.item(), 100 * (meter.mfu() or 0))
        if ckpt.should_checkpoint():
            engine.save_checkpoint(ckpt.out_dir, tag=f'step_{step}')

def build_gpt(model_cfg): ...
def build_dataloader(train_cfg): ...
def estimate_flops(cfg): ...

if __name__ == '__main__':
    main()


## 2. End-to-End Launch Sequence

```bash
# 0. (one-time) Build & push the image
make docker-build
docker tag hpc-llm-forge:latest ghcr.io/HAYDARKILIC/hpc-llm-forge:v0.1.0
docker push ghcr.io/HAYDARKILIC/hpc-llm-forge:v0.1.0

# 1. Launch on RunPod (1 command)
export RUNPOD_API_KEY=...
export WANDB_API_KEY=...
make launch-cloud PROVIDER=runpod GPU=H100 NUM_GPUS=8 \\
    CONFIG=configs/training/gpt_1b3.yaml
```

The `scripts/launch_cloud.sh` wrapper:

1. Validates the environment.
2. Calls `hpcllmforge.orchestration.cloud_launcher.CloudLauncher`.
3. Streams the pod's log to stdout.
4. Opens the WandB run URL in a browser when the first metric is logged.


## 3. Validation Checklist

A successful capstone run satisfies every box below:

- [ ] **Determinism.** Two consecutive runs with identical seed and `deterministic_algorithms=True` produce bit-exact loss curves for the first 100 steps.
- [ ] **MFU.** ≥ 40% sustained MFU on H100 BF16.
- [ ] **Memory.** Per-GPU peak memory matches the ZeRO-3 prediction within 5%.
- [ ] **Resilience.** A simulated preemption (SIGTERM at step 1,000) resumes from the most recent checkpoint with no loss discontinuity.
- [ ] **Reproducibility.** `hpcllmforge.utils.topology.fingerprint()` is recorded in every WandB run.


## 4. Where to go next

* **Mixture-of-Experts.** Replace dense MLP blocks with sparsely-gated experts. Requires expert-parallelism on top of ZeRO + TP + PP.
* **Sequence parallelism.** Extend tensor parallelism to also partition the sequence dimension, reducing activation memory in long-context training.
* **FP8 end-to-end.** Quantize forward, backward, *and* optimizer state to FP8 (E4M3 / E5M2) — the H100 / B200 roadmap.
* **Inference serving.** Deploy the trained model with vLLM or TensorRT-LLM and measure tokens-per-second-per-dollar.
